# Imports

## Standard Library

In [54]:
import os          # file paths, directory management
import sys         # system-level operations (rarely used, but useful for debugging)
import time        # timing execution (performance measurement)
import gc          # garbage collection (manual memory cleanup)
import csv         # CSV file handling (reading/writing tabular data)

## Data Handling & Numerical Computation

In [55]:
import numpy as np         # numerical operations, arrays, embeddings
import pandas as pd        # dataset loading and manipulation (DataFrames)

## System Monitoring & External Communication

In [56]:
import psutil              # monitor RAM/CPU usage (GreenAI metrics)
import requests            # send notifications (by NTFY)

## Image Processing & Computer Vision

In [57]:
from PIL import Image      # image loading
import cv2                 # OpenCV (advanced image processing if needed)

## Progress & Visualization

In [58]:
from tqdm.notebook import tqdm   # progress bars

import matplotlib.pyplot as plt  # plotting results
import seaborn as sns            # statistical visualizations

## Statistics / Evaluation

In [59]:
from scipy.stats import spearmanr   # rank correlation (similarity preservation)
import pickle 

## PyTorch (Core Deep Learning Framework)

In [60]:
import torch
import torch.nn as nn              # neural network layers

## TorchVision (Pretrained Vision Models & Transforms)

In [61]:
from torchvision import models, transforms

## Transformers (HuggingFace Models)

In [62]:
from transformers import (
    # Vision Transformers
    ViTModel, ViTImageProcessor,
    DeiTModel, DeiTImageProcessor,

    # Generic auto models (flexible loading)
    AutoImageProcessor, AutoModel,

    # CLIP (multimodal)
    CLIPProcessor, CLIPVisionModel, CLIPTextModel,

    # Text models
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model
)

## Explainable Libraries

In [63]:
import captum
from captum.attr import IntegratedGradients, Saliency, Occlusion, DeepLift, GradientShap

import shap

import quantus

In [64]:
import urllib.request
import zipfile
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configuration

## Dataset Selection

In [65]:
# Available datasets: Flickr8k, Flickr30k, ConceptualCaptions
CURRENT_DATASET = "Flickr8k" 
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]

assert CURRENT_DATASET in ALL_DATASETS, f"{CURRENT_DATASET} is not a valid dataset"


## Directory Architecture

In [66]:
BASE_DIR = os.path.join(os.getcwd(), 'TFE_Data')

DATASETS_DIR = os.path.join(BASE_DIR, 'Datasets')
#RAW_DIR = os.path.join(BASE_DIR, 'Features_RAW', CURRENT_DATASET)
#RESULTS_DIR = os.path.join(BASE_DIR, 'Unimodal_VisionXAI_Results', CURRENT_DATASET)
"""

def get_raw_dir(dataset, modality):
    path = os.path.join(BASE_DIR, "Features", "raw", dataset, modality)
    os.makedirs(path, exist_ok=True)
    return path

def get_indexed_dir(dataset, modality, model):
    path = os.path.join('TFE_Data', "Results_Unimodal", dataset, modality, model)
    os.makedirs(path, exist_ok=True)
    return path
    
def load_embeddings(dataset, modality, model):
    file_path = os.path.join('TFE_Data', "Results_Unimodal", dataset, modality, model, "image_tensors.npy")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing RAW image_tensors: {file_path}")
    return np.load(file_path)

def get_results_dir(dataset):
    path = os.path.join(BASE_DIR, "Unimodal_VisionXAI_Results", dataset)
    os.makedirs(path, exist_ok=True)
    return path

"""

'\n\ndef get_raw_dir(dataset, modality):\n    path = os.path.join(BASE_DIR, "Features", "raw", dataset, modality)\n    os.makedirs(path, exist_ok=True)\n    return path\n\ndef get_indexed_dir(dataset, modality, model):\n    path = os.path.join(\'TFE_Data\', "Results_Unimodal", dataset, modality, model)\n    os.makedirs(path, exist_ok=True)\n    return path\n\ndef load_embeddings(dataset, modality, model):\n    file_path = os.path.join(\'TFE_Data\', "Results_Unimodal", dataset, modality, model, "image_tensors.npy")\n    if not os.path.exists(file_path):\n        raise FileNotFoundError(f"Missing RAW image_tensors: {file_path}")\n    return np.load(file_path)\n\ndef get_results_dir(dataset):\n    path = os.path.join(BASE_DIR, "Unimodal_VisionXAI_Results", dataset)\n    os.makedirs(path, exist_ok=True)\n    return path\n\n'

In [67]:

def get_indexed_dir(dataset, modality, model):
    path = os.path.join('TFE_Data', "Results_Unimodal", dataset, modality, model)
    os.makedirs(path, exist_ok=True)
    return path

## Device Configuration

In [68]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Environment initialized. Using device: {device}")

Environment initialized. Using device: cuda


## Notification Setup

In [69]:
def send_ntfy_notification(message, title="TFE Pipeline Update"):
    """Sends a push notification via ntfy.sh."""
    NTFY_TOPIC = "aysel_tfe_server_9988"
    try:
        requests.post(
            f"https://ntfy.sh/{'aysel_tfe_server_9988'}",
            data=message.encode(encoding='utf-8'),
            headers={"Title": title}
        )
    except Exception as e:
        print(f"Notification failed: {e}")



# Data Loading

In [70]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {k: v.squeeze(0) for k, v in enc.items()}


### Text Models

In [71]:
from transformers import CLIPTokenizer


TEXT_MODELS = {
    "bert": (BertTokenizer.from_pretrained("bert-base-uncased"),
             BertModel.from_pretrained("bert-base-uncased")),
    "roberta": (RobertaTokenizer.from_pretrained("roberta-base"),
                RobertaModel.from_pretrained("roberta-base")),
    "gpt2": (GPT2Tokenizer.from_pretrained("gpt2"),
             GPT2Model.from_pretrained("gpt2")),
    "clip_text": (
        CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32"),
        CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
)
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
visual_projection.weight                                       | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.

# Explanation 

## Captum

### Saliency 

In [72]:

def explain_saliency(model, input_ids, attention_mask, label):
    sal = Saliency(model)
    attr = sal.attribute(
        input_ids.unsqueeze(0),
        additional_forward_args=attention_mask.unsqueeze(0),
        target=label
    )
    return attr.squeeze(0).sum(dim=-1)

### Integrated Gradient

In [73]:
def explain_ig(model, input_ids, attention_mask, label):
    ig = IntegratedGradients(model)
    attr = ig.attribute(
        inputs=input_ids.unsqueeze(0),
        additional_forward_args=attention_mask.unsqueeze(0),
        target=label
    )
    return attr.squeeze(0).sum(dim=-1)

### GradientSHAP

In [74]:
def explain_gradientShap(model, input_ids, attention_mask, label):
    gs = GradientShap(model)
    baseline = torch.zeros_like(input_ids)
    attr = gs.attribute(
        input_ids.unsqueeze(0),
        baselines=baseline.unsqueeze(0),
        additional_forward_args=attention_mask.unsqueeze(0),
        target=label
    )
    return attr.squeeze(0).sum(dim=-1)


### DeepLift

In [75]:
def explain_deeplift(model, input_ids, attention_mask, label):
    dl = DeepLift(model)
    attr = dl.attribute(
        input_ids.unsqueeze(0),
        additional_forward_args=attention_mask.unsqueeze(0),
        target=label
    )
    return attr.squeeze(0).sum(dim=-1)

## SHAP Explanations

In [76]:
class TextXAIWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        hidden = model.config.hidden_size
        self.head = nn.Linear(hidden, 1000)  # mirror vision pipeline

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.head(cls)
        return logits


In [77]:
def explain_shap(model, input_ids, attention_mask, background):
    explainer = shap.GradientExplainer(model, background)
    shap_vals = explainer.shap_values(
        [input_ids.unsqueeze(0), attention_mask.unsqueeze(0)]
    )
    return torch.tensor(shap_vals[0]).squeeze(0).sum(dim=-1)

# Evaluation

## Quantus

In [78]:
from quantus.metrics.faithfulness.infidelity import Infidelity
from quantus.metrics.robustness.max_sensitivity import MaxSensitivity
from quantus.metrics.complexity.sparseness import Sparseness
from quantus.metrics.faithfulness.sensitivity_n import SensitivityN

In [79]:
def evaluate_quantus(model, img, label, attr):
    metrics = [
        Infidelity(),
        MaxSensitivity(),
        Sparseness(),
        SensitivityN()
    ]

    results = {}
    for m in metrics:
        results[m.__class__.__name__] = m(
            model=model,
            x=img.unsqueeze(0),
            a=attr.unsqueeze(0),
            y=torch.tensor([label])
        )
    return results

# Save Results

In [80]:
def save_attribution_tensor(tensor, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    torch.save(tensor.cpu(), out_path)


In [81]:
def save_quantus_csv(all_metrics, out_path="TFE_Data/Unimodal/XAI/vision/quantus_results.csv"):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["dataset", "model", "image_index", "method", "metric", "value"])

        for entry in all_metrics:
            dataset = entry["dataset"]
            model = entry["model"]
            results = entry["results"]

            for idx, res in enumerate(results):
                for method, metrics_dict in res.items():
                    for metric_name, metric_value in metrics_dict.items():
                        writer.writerow([dataset, model, idx, method, metric_name, metric_value])

# Visualisation

## Heatmap Visualisation (Captum and SHAP)

In [82]:
def visualize_heatmap(tokens, scores, title):
    plt.figure(figsize=(12, 1.5))
    plt.bar(range(len(tokens)), scores)
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.title(title)
    plt.tight_layout()
    plt.show()

    sns.heatmap(scores[np.newaxis, :], cmap="coolwarm")
    plt.show()

## SHAP visualisation

In [83]:
def visualize_shap(shap_values, original_img):
    """
    shap_values: tensor [1,3,H,W] or [3,H,W]
    """
    vals = shap_values.squeeze().detach().cpu().numpy()
    vals = np.mean(vals, axis=0)

    img = original_img.detach().cpu().numpy().transpose(1,2,0)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.imshow(vals, cmap='coolwarm', alpha=0.45)
    plt.title("SHAP Attribution")
    plt.axis("off")
    plt.show()


## Side by Side for each image

In [84]:
def visualize_side_by_side(original_img, ig, sal, gs, occ, shap_vals, title="XAI Comparison"):
    """
    Shows 6 panels:
    - Original
    - IG
    - Saliency
    - GradientShap
    - Occlusion
    - SHAP
    """

    # Convert tensors to numpy
    img_np = original_img.detach().cpu().numpy().transpose(1,2,0)
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

    def prep(attr):
        a = attr.detach().cpu().numpy()
        if a.ndim == 3:
            a = a.mean(axis=0)
        return a

    ig_np   = prep(ig)
    sal_np  = prep(sal)
    gs_np   = prep(gs)
    occ_np  = prep(occ)
    shap_np = prep(shap_vals.squeeze())

    fig, axs = plt.subplots(2, 3, figsize=(14, 9))
    axs = axs.flatten()

    axs[0].imshow(img_np)
    axs[0].set_title("Original")
    axs[0].axis("off")

    axs[1].imshow(img_np)
    axs[1].imshow(ig_np, cmap="jet", alpha=0.45)
    axs[1].set_title("Integrated Gradients")
    axs[1].axis("off")

    axs[2].imshow(img_np)
    axs[2].imshow(sal_np, cmap="jet", alpha=0.45)
    axs[2].set_title("Saliency")
    axs[2].axis("off")

    axs[3].imshow(img_np)
    axs[3].imshow(gs_np, cmap="jet", alpha=0.45)
    axs[3].set_title("GradientShap")
    axs[3].axis("off")

    axs[4].imshow(img_np)
    axs[4].imshow(occ_np, cmap="jet", alpha=0.45)
    axs[4].set_title("Occlusion")
    axs[4].axis("off")

    axs[5].imshow(img_np)
    axs[5].imshow(shap_np, cmap="coolwarm", alpha=0.45)
    axs[5].set_title("SHAP")
    axs[5].axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


## Per model summary

In [85]:
def summarize_model_attributions(model_name, all_attrs):
    avg = np.mean(np.stack(all_attrs), axis=0)
    plt.figure(figsize=(12, 2))
    sns.heatmap(avg[np.newaxis, :], cmap="coolwarm")
    plt.title(f"{model_name} — Average Token Attribution")
    plt.show()

# Execution

## Debug of Full Mode

In [86]:
MODE = "C"

if MODE == "A":
    MAX_SHAP_IMAGES = None
    MAX_QUANTUS_IMAGES = None
    MAX_VISUAL_IMAGES = None
elif MODE == "B":
    MAX_SHAP_IMAGES = 32
    MAX_QUANTUS_IMAGES = 100
    MAX_VISUAL_IMAGES = 20
elif MODE == "C":
    MAX_SHAP_IMAGES = 8
    MAX_QUANTUS_IMAGES = 20
    MAX_VISUAL_IMAGES = 5



In [87]:
def run_xai(model_name, texts, tokenizer, base_model, device, dataset_name):
    wrapper = TextXAIWrapper(base_model).to(device)
    dataset = TextDataset(texts, tokenizer)
    loader = torch.utils.data.DataLoader(dataset, batch_size=1)

    # SHAP background
    background_input_ids = []
    background_attention = []

    for i, batch in enumerate(loader):
        if i >= 8: break
        background_input_ids.append(batch['input_ids'].to(device))
        background_attention.append(batch['attention_mask'].to(device))

    background = [
        torch.stack(background_input_ids),
        torch.stack(background_attention)
    ]


    all_attrs = []
    quantus_results = []

    for idx, batch in enumerate(tqdm(loader, desc=f"{model_name} XAI")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = wrapper(input_ids, attention_mask)
        label = logits.argmax(dim=1).item()

        # Captum methods
        ig_attr = explain_ig(wrapper, input_ids, attention_mask, label)
        sal_attr = explain_saliency(wrapper, input_ids, attention_mask, label)
        gs_attr = explain_gradientShap(wrapper, input_ids, attention_mask, label)
        dl_attr = explain_deeplift(wrapper, input_ids, attention_mask, label)

        # SHAP
        if idx < MAX_SHAP_IMAGES:
            shap_attr = explain_shap(wrapper, input_ids, attention_mask, background)
        else:
            shap_attr = torch.zeros_like(ig_attr)

        # Visualization
        if idx < MAX_VISUAL_IMAGES:
            tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
            visualize_heatmap(tokens, ig_attr.cpu().numpy(), f"{model_name} IG — {idx}")

        # Collect
        all_attrs.append(ig_attr.cpu().numpy())

        # Quantus
        if idx < MAX_QUANTUS_IMAGES:
            q = evaluate_quantus(wrapper, input_ids, attention_mask, label, ig_attr)
            quantus_results.append(q)

    summarize_model_attributions(model_name, all_attrs)
    return quantus_results


In [88]:
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]
all_metrics = []

for ds_name in ALL_DATASETS:
    df_path = os.path.join(DATASETS_DIR, f"df_{ds_name}.pkl")
    if not os.path.exists(df_path):
        print(f"Skipping {ds_name}: no metadata")
        continue

    df = pd.read_pickle(df_path)
    texts = df["caption"].tolist()

    for model_name, (tokenizer, base_model) in TEXT_MODELS.items():
        print(f"\n=== Running {model_name} on {ds_name} ===")
        results = run_xai(
            model_name,
            texts,
            tokenizer,
            base_model.to(device),
            device,
            ds_name
        )
        all_metrics.append({
            "dataset": ds_name,
            "model": model_name,
            "results": results
        })

KeyError: 'caption'